# 🛒 REES46 eCommerce Behavior Data — Deep-Dive Exploratory Data Analysis

> **Dataset:** [REES46 eCommerce Behavior Data from Multi Category Store](https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store)  
> **Objective:** Analyze multi-category clickstream behavior with a focus on **Cart Abandonment** — uncovering behavioral patterns, funnel drop-offs, temporal dynamics, and leakage-free feature engineering signals for downstream Tabular ML models.

---

## 📋 Dataset Overview

| Attribute | Details |
|---|---|
| **Source** | REES46 Marketing Platform / Open CDP |
| **Time Span** | October & November 2019 |
| **Scale** | ~109.95 million events (~13.7 GB uncompressed) |
| **Domain** | Multi-category online store (electronics, appliances, furniture, apparel) |
| **Columns (9)** | `event_time`, `event_type`, `product_id`, `category_id`, `category_code`, `brand`, `price`, `user_id`, `user_session` |
| **Event Types (4)** | `view`, `cart`, `remove_from_cart`, `purchase` |

### Analysis Objectives:
1. **Data Quality & Schema Profiling** — Data types, missing value patterns, cardinality
2. **Conversion Funnel Analysis** — `View → Cart → Purchase` drop-off rates
3. **Cart Abandonment Deep Dive** — Session-level & user-level abandonment behavior
4. **Price Sensitivity Analysis** — Price distributions across abandoned vs. converted carts
5. **Temporal Dynamics** — Hourly, day-of-week, and daily abandonment trends
6. **Category & Brand Breakdown** — Abandonment variance across product verticals
7. **User Segmentation & Feature Engineering Preview** — Preparing inputs for `IntentClassifier`, `ChurnPredictor`, and `TimingOptimizer`

## 1. 🔧 Setup & Imports

In [ ]:
# Uncomment to install dependencies in a fresh environment
# !pip install pandas numpy matplotlib seaborn plotly scikit-learn pyarrow tqdm

import warnings
from collections import Counter
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

# Global plot styling
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_rows', 100)

# Consistent color palette across charts
COLORS = {
    'view': '#3498db',
    'cart': '#f39c12',
    'purchase': '#27ae60',
    'remove_from_cart': '#e74c3c',
    'primary': '#2c3e50',
    'secondary': '#8e44ad',
    'accent': '#e67e22',
}

print('All libraries imported successfully.')
print(f'Analysis timestamp: {datetime.now().strftime("%Y-%m-%d %H:%M")}')

## 2. 📂 Data Loading

> **⚠️ Memory Note:** The combined uncompressed CSVs are ~13.7 GB. On machines with < 32 GB RAM, load a single month, use down-cast dtypes below, or enable the optional sampling cell.

### Downloading via Project CLI or Kaggle CLI:
```bash
# Option 1: Project helper script (from fine-tuning/ root)
python scripts/download_datasets.py --dataset rees46

# Option 2: Quick synthetic sample with identical schema for smoke testing
python scripts/download_datasets.py --synthetic-demo
```

In [ ]:
import glob

DATA_DIR = '../data/raw/rees46/'

# Memory-optimized column dtypes
DTYPES = {
    'event_type': 'category',
    'product_id': 'int32',
    'category_id': 'int64',
    'category_code': 'str',
    'brand': 'str',
    'price': 'float32',
    'user_id': 'int32',
    'user_session': 'str',
}

PARSE_DATES = ['event_time']
csv_files = sorted(glob.glob(f'{DATA_DIR}*.csv'))
print(f'Discovered {len(csv_files)} CSV file(s) in {DATA_DIR}: {csv_files}')

In [ ]:
# Load all available REES46 CSV files (full Oct/Nov CSVs or sample CSV)
frames = []
for csv_path in csv_files:
    print(f'Loading {csv_path} ...')
    part = pd.read_csv(csv_path, dtype=DTYPES, parse_dates=PARSE_DATES)
    print(f'  -> {part.shape[0]:,} rows | {part.memory_usage(deep=True).sum() / 1e9:.2f} GB')
    frames.append(part)

df = pd.concat(frames, ignore_index=True)
del frames

print('\nCombined Dataset Summary:')
print(f'  Total rows   : {df.shape[0]:,}')
print(f'  Total columns: {df.shape[1]}')
print(f'  Memory usage : {df.memory_usage(deep=True).sum() / 1e9:.2f} GB')
print(f'  Date range   : {df["event_time"].min()} -> {df["event_time"].max()}')

> **💡 Optional Down-Sampling for Constrained RAM (< 16 GB)**
>
> Uncomment the cell below to cap the in-memory DataFrame at 10M events:

In [ ]:
# SAMPLE_SIZE = 10_000_000
# if len(df) > SAMPLE_SIZE:
#     df = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
#     print(f'Sampled down to {SAMPLE_SIZE:,} rows')

## 3. 🔍 Data Overview & Quality Check

In [ ]:
print('DATASET SCHEMA & MEMORY PROFILE')
print('=' * 60)
df.info(memory_usage='deep')
display(df.head(10))

In [ ]:
print('NUMERIC DESCRIPTIVE STATISTICS')
print('=' * 60)
display(df.describe())

In [ ]:
# Missing values analysis
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    'Column': missing.index,
    'Missing Count': missing.values,
    'Missing %': missing_pct.values,
    'Non-Null Count': len(df) - missing.values,
    'Dtype': [str(df[col].dtype) for col in df.columns],
})

display(missing_df)

fig, ax = plt.subplots(figsize=(12, 5))
bar_colors = ['#e74c3c' if pct > 0 else '#27ae60' for pct in missing_pct.values]
bars = ax.bar(missing_df['Column'], missing_df['Missing %'], color=bar_colors, edgecolor='white')
for bar, pct in zip(bars, missing_pct.values):
    if pct > 0:
        ax.text(
            bar.get_x() + bar.get_width() / 2.0,
            bar.get_height() + 0.5,
            f'{pct:.1f}%',
            ha='center',
            va='bottom',
            fontweight='bold',
        )
ax.set_title('Missing Value Percentage by Column', fontsize=15, fontweight='bold')
ax.set_ylabel('Missing (%)')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

print('Observation: Only category_code and brand contain missing values (accessories / unbranded items).')

In [ ]:
unique_counts = df.nunique()
print('CARDINALITY PER COLUMN')
print('=' * 60)
for col in df.columns:
    print(f'  {col:20s} -> {unique_counts[col]:>12,} unique values')

## 4. 📊 Event Type Distribution

Examine the proportion of `view`, `cart`, `remove_from_cart`, and `purchase` events.

In [ ]:
event_counts = df['event_type'].value_counts()
event_pct = df['event_type'].value_counts(normalize=True) * 100

event_summary = pd.DataFrame({
    'Event Type': event_counts.index,
    'Count': event_counts.values,
    'Percentage (%)': event_pct.values,
})
display(event_summary)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
event_colors = [COLORS.get(str(et), '#95a5a6') for et in event_counts.index]

bars = axes[0].bar(event_counts.index.astype(str), event_counts.values, color=event_colors, edgecolor='white')
for bar, count, pct in zip(bars, event_counts.values, event_pct.values):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2.0,
        bar.get_height(),
        f'{count:,.0f}\n({pct:.1f}%)',
        ha='center',
        va='bottom',
        fontsize=10,
        fontweight='bold',
    )
axes[0].set_title('Event Counts by Type', fontweight='bold')
axes[0].set_ylabel('Event Count')

axes[1].pie(
    event_counts.values,
    labels=event_counts.index.astype(str),
    colors=event_colors,
    autopct='%1.1f%%',
    startangle=90,
    pctdistance=0.75,
    wedgeprops=dict(width=0.5, edgecolor='white', linewidth=2),
)
axes[1].set_title('Event Type Share', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. 🔄 Conversion Funnel Analysis — View → Cart → Purchase

Quantify user progression and stage-by-stage drop-off across the shopping funnel.

In [ ]:
users_viewed = df[df['event_type'] == 'view']['user_id'].nunique()
users_carted = df[df['event_type'] == 'cart']['user_id'].nunique()
users_purchased = df[df['event_type'] == 'purchase']['user_id'].nunique()
users_removed = df[df['event_type'] == 'remove_from_cart']['user_id'].nunique()
total_users = df['user_id'].nunique()

funnel_data = pd.DataFrame({
    'Stage': ['View Product', 'Add to Cart', 'Purchase'],
    'Unique Users': [users_viewed, users_carted, users_purchased],
    '% of Total Users': [
        users_viewed / max(total_users, 1) * 100,
        users_carted / max(total_users, 1) * 100,
        users_purchased / max(total_users, 1) * 100,
    ],
    '% of Previous Stage': [
        100.0,
        users_carted / max(users_viewed, 1) * 100,
        users_purchased / max(users_carted, 1) * 100,
    ],
})
display(funnel_data)

print(f'User-level Cart Abandonment Rate: {(1 - users_purchased / max(users_carted, 1)) * 100:.2f}%')
print(f'Cart users who triggered remove_from_cart: {users_removed:,}')

In [ ]:
fig = go.Figure(go.Funnel(
    y=['View Product', 'Add to Cart', 'Purchase'],
    x=[users_viewed, users_carted, users_purchased],
    textposition='inside',
    textinfo='value+percent initial+percent previous',
    marker=dict(color=['#3498db', '#f39c12', '#27ae60']),
))
fig.update_layout(
    title='User Conversion Funnel — REES46',
    height=420,
    width=820,
)
fig.show()

In [ ]:
events_view = int(event_counts.get('view', 0))
events_cart = int(event_counts.get('cart', 0))
events_purchase = int(event_counts.get('purchase', 0))
events_remove = int(event_counts.get('remove_from_cart', 0))

print('EVENT-LEVEL FUNNEL SUMMARY')
print('=' * 60)
print(f'  Views            : {events_view:>12,}')
print(f'  Add to Cart      : {events_cart:>12,} ({events_cart / max(events_view, 1) * 100:.2f}% of views)')
print(f'  Remove from Cart : {events_remove:>12,} ({events_remove / max(events_cart, 1) * 100:.2f}% of carts)')
print(f'  Purchases        : {events_purchase:>12,} ({events_purchase / max(events_cart, 1) * 100:.2f}% of carts)')

## 6. 🛒 Cart Abandonment — Session & User Deep Dive

Measure cart abandonment at both the individual session level and lifetime user level.

In [ ]:
cart_sessions = set(df[df['event_type'] == 'cart']['user_session'].dropna().unique())
purchase_sessions = set(df[df['event_type'] == 'purchase']['user_session'].dropna().unique())

abandoned_sessions = cart_sessions - purchase_sessions
converted_sessions = cart_sessions & purchase_sessions

total_cart_sessions = max(len(cart_sessions), 1)
print('SESSION-LEVEL CART ABANDONMENT')
print('=' * 60)
print(f'  Sessions with cart event : {len(cart_sessions):,}')
print(f'  Converted sessions       : {len(converted_sessions):,} ({len(converted_sessions) / total_cart_sessions * 100:.1f}%)')
print(f'  Abandoned sessions       : {len(abandoned_sessions):,} ({len(abandoned_sessions) / total_cart_sessions * 100:.1f}%)')

fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(
    [len(abandoned_sessions), len(converted_sessions)],
    labels=[f'Abandoned\n({len(abandoned_sessions):,})', f'Converted\n({len(converted_sessions):,})'],
    colors=['#e74c3c', '#27ae60'],
    explode=(0.04, 0),
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops=dict(width=0.5, edgecolor='white', linewidth=2),
)
ax.set_title('Session-Level Cart Abandonment Rate', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
users_ever_carted = set(df[df['event_type'] == 'cart']['user_id'].unique())
users_ever_purchased = set(df[df['event_type'] == 'purchase']['user_id'].unique())

pure_abandoners = users_ever_carted - users_ever_purchased
eventual_buyers = users_ever_carted & users_ever_purchased

print('USER-LEVEL CART OUTCOME')
print('=' * 60)
print(f'  Users who ever added to cart : {len(users_ever_carted):,}')
print(f'  Purchased at least once      : {len(eventual_buyers):,} ({len(eventual_buyers) / max(len(users_ever_carted), 1) * 100:.1f}%)')
print(f'  Never purchased (Pure Abandon): {len(pure_abandoners):,} ({len(pure_abandoners) / max(len(users_ever_carted), 1) * 100:.1f}%)')

## 7. 💰 Price Sensitivity Analysis

Compare item and basket price distributions across abandoned vs. purchased carts.

In [ ]:
print('OVERALL PRICE SUMMARY')
print(df['price'].describe())

print('\nPRICE STATISTICS BY EVENT TYPE')
price_by_event = df.groupby('event_type', observed=False)['price'].agg(['mean', 'median', 'std', 'min', 'max'])
display(price_by_event)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

df_nonzero = df[df['price'] > 0]
upper_q = df_nonzero['price'].quantile(0.99)
axes[0].hist(df_nonzero[df_nonzero['price'] <= upper_q]['price'], bins=80, color=COLORS['primary'], alpha=0.75, edgecolor='white')
axes[0].set_title('Item Price Distribution (<= 99th Percentile)', fontweight='bold')
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Event Frequency')

event_order = [e for e in ['view', 'cart', 'purchase', 'remove_from_cart'] if e in df_nonzero['event_type'].unique()]
sampled_groups = [
    df_nonzero[df_nonzero['event_type'] == e]['price'].sample(
        min(10000, len(df_nonzero[df_nonzero['event_type'] == e])), random_state=42
    ).values
    for e in event_order
]
bp = axes[1].boxplot(sampled_groups, labels=event_order, patch_artist=True, showfliers=False)
for patch, e in zip(bp['boxes'], event_order):
    patch.set_facecolor(COLORS.get(e, '#95a5a6'))
    patch.set_alpha(0.75)
axes[1].set_title('Item Price Spread by Event Type', fontweight='bold')
axes[1].set_ylabel('Price ($)')
plt.tight_layout()
plt.show()

In [ ]:
cart_events = df[df['event_type'] == 'cart'].copy()
cart_events['outcome'] = np.where(
    cart_events['user_session'].isin(converted_sessions), 'Purchased', 'Abandoned'
)
display(cart_events.groupby('outcome')['price'].agg(['mean', 'median', 'std', 'count']))

cart_totals = cart_events.groupby(['user_session', 'outcome'], as_index=False).agg(
    total_cart_value=('price', 'sum'),
    items_in_cart=('price', 'count'),
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
cap = cart_events['price'].quantile(0.95)
for label, col in [('Abandoned', '#e74c3c'), ('Purchased', '#27ae60')]:
    sub = cart_events[(cart_events['outcome'] == label) & (cart_events['price'] <= cap)]['price']
    if len(sub) > 0:
        axes[0].hist(sub, bins=45, alpha=0.55, density=True, label=label, color=col)
axes[0].set_title('Carted Item Price Density: Abandoned vs. Purchased', fontweight='bold')
axes[0].set_xlabel('Price ($)')
axes[0].legend()

sns.boxplot(data=cart_totals, x='outcome', y='total_cart_value', showfliers=False, ax=axes[1], palette=['#e74c3c', '#27ae60'])
axes[1].set_title('Session Total Cart Value: Abandoned vs. Purchased', fontweight='bold')
axes[1].set_ylabel('Total Cart Value ($)')
plt.tight_layout()
plt.show()

## 8. ⏰ Temporal Analysis — Hourly, Weekly & Daily Trends

Identify peak shopping hours and temporal abandonment patterns to inform the `TimingOptimizer` survival model.

In [ ]:
df['hour'] = df['event_time'].dt.hour
df['day_of_week'] = df['event_time'].dt.dayofweek
df['day_name'] = df['event_time'].dt.day_name()
df['date'] = df['event_time'].dt.date
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
print('Extracted temporal features: hour, day_of_week, day_name, date, is_weekend.')

In [ ]:
hourly = df.groupby(['hour', 'event_type'], observed=False).size().unstack(fill_value=0)

fig, axes = plt.subplots(2, 1, figsize=(16, 11))
for et in ['view', 'cart', 'purchase', 'remove_from_cart']:
    if et in hourly.columns:
        axes[0].plot(hourly.index, hourly[et], label=et, color=COLORS[et], linewidth=2.2, marker='o')
axes[0].set_title('Hourly Event Volume by Type (UTC)', fontweight='bold')
axes[0].set_xticks(range(24))
axes[0].legend()

hourly_cart_sess = df[df['event_type'] == 'cart'].groupby('hour')['user_session'].nunique()
hourly_purch_sess = df[df['event_type'] == 'purchase'].groupby('hour')['user_session'].nunique()
hourly_abandon_pct = ((hourly_cart_sess - hourly_purch_sess.reindex(hourly_cart_sess.index, fill_value=0)) / hourly_cart_sess.clip(lower=1) * 100)

axes[1].bar(hourly_abandon_pct.index, hourly_abandon_pct.values, color='#e74c3c', alpha=0.8)
axes[1].axhline(hourly_abandon_pct.mean(), color='black', linestyle='--', label=f'Mean: {hourly_abandon_pct.mean():.1f}%')
axes[1].set_title('Hourly Cart Abandonment Rate (%)', fontweight='bold')
axes[1].set_xlabel('Hour of Day (UTC)')
axes[1].set_ylabel('Abandonment Rate (%)')
axes[1].set_xticks(range(24))
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily_dow = df.groupby(['day_name', 'event_type'], observed=False).size().unstack(fill_value=0).reindex(day_order, fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
cols_present = [c for c in ['view', 'cart', 'purchase'] if c in daily_dow.columns]
daily_dow[cols_present].plot(kind='bar', ax=axes[0], color=[COLORS[c] for c in cols_present])
axes[0].set_title('Event Volume by Day of Week', fontweight='bold')
axes[0].tick_params(axis='x', rotation=35)

conv_dow = (daily_dow.get('purchase', 0) / daily_dow.get('cart', 1).replace(0, 1)) * 100
axes[1].bar(day_order, conv_dow.values, color='#27ae60', alpha=0.85)
axes[1].set_title('Cart-to-Purchase Conversion Rate (%) by Day of Week', fontweight='bold')
axes[1].tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.show()

In [ ]:
daily_ts = df.groupby(['date', 'event_type'], observed=False).size().unstack(fill_value=0)

fig, axes = plt.subplots(2, 1, figsize=(16, 9))
for et in ['view', 'cart', 'purchase']:
    if et in daily_ts.columns:
        axes[0].plot(daily_ts.index, daily_ts[et], label=et, color=COLORS[et], linewidth=2)
axes[0].set_title('Daily Event Volume Over Time', fontweight='bold')
axes[0].legend()

daily_abandon = (1.0 - daily_ts.get('purchase', 0) / daily_ts.get('cart', 1).replace(0, 1)) * 100
axes[1].plot(daily_abandon.index, daily_abandon.values, color='#e74c3c', linewidth=2)
axes[1].fill_between(daily_abandon.index, daily_abandon.values, alpha=0.25, color='#e74c3c')
axes[1].set_title('Daily Cart Abandonment Rate (%)', fontweight='bold')
axes[1].set_ylabel('Abandonment (%)')
plt.tight_layout()
plt.show()

## 9. 📦 Category & Brand Performance

Compare conversion and cart abandonment rates across top product categories and brands.

In [ ]:
df['category_main'] = df['category_code'].fillna('unknown').apply(
    lambda x: x.split('.')[0] if isinstance(x, str) and x.strip() else 'unknown'
)
top_categories = df['category_main'].value_counts().head(12)
print('Top Product Categories by Event Volume:')
display(top_categories.to_frame('event_count'))

In [ ]:
cat_rows = []
for cat in top_categories.index:
    sub = df[df['category_main'] == cat]
    views = int((sub['event_type'] == 'view').sum())
    carts = int((sub['event_type'] == 'cart').sum())
    purchases = int((sub['event_type'] == 'purchase').sum())
    cat_rows.append({
        'Category': cat,
        'Views': views,
        'Carts': carts,
        'Purchases': purchases,
        'View->Cart %': carts / max(views, 1) * 100,
        'Cart->Purchase %': purchases / max(carts, 1) * 100,
        'Abandon Rate %': (1 - purchases / max(carts, 1)) * 100 if carts > 0 else 0.0,
        'Avg Price ($)': float(sub['price'].mean()),
    })
cat_df_analysis = pd.DataFrame(cat_rows)
display(cat_df_analysis.round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
cat_sorted = cat_df_analysis.sort_values('Abandon Rate %')
axes[0].barh(cat_sorted['Category'], cat_sorted['Abandon Rate %'], color='#e74c3c', alpha=0.85)
axes[0].set_title('Cart Abandonment Rate (%) by Category', fontweight='bold')

cat_sorted2 = cat_df_analysis.sort_values('View->Cart %')
axes[1].barh(cat_sorted2['Category'], cat_sorted2['View->Cart %'], color=COLORS['cart'], alpha=0.85)
axes[1].set_title('View-to-Cart Rate (%) by Category', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
valid_brands = df[df['brand'].notna() & (df['brand'].str.strip() != '')]
top_brands = valid_brands['brand'].value_counts().head(15).index

brand_rows = []
for b in top_brands:
    sub = valid_brands[valid_brands['brand'] == b]
    carts = int((sub['event_type'] == 'cart').sum())
    purchases = int((sub['event_type'] == 'purchase').sum())
    brand_rows.append({
        'Brand': b,
        'Events': len(sub),
        'Carts': carts,
        'Purchases': purchases,
        'Abandon Rate %': (1 - purchases / max(carts, 1)) * 100 if carts > 0 else 0.0,
        'Avg Price ($)': float(sub['price'].mean()),
    })
brand_df_analysis = pd.DataFrame(brand_rows)
display(brand_df_analysis.round(2))

fig, ax = plt.subplots(figsize=(12, 6))
b_sorted = brand_df_analysis.sort_values('Abandon Rate %')
ax.barh(b_sorted['Brand'], b_sorted['Abandon Rate %'], color='#8e44ad', alpha=0.85)
ax.set_title('Cart Abandonment Rate (%) Across Top Brands', fontweight='bold')
ax.set_xlabel('Abandonment Rate (%)')
plt.tight_layout()
plt.show()

## 10. 👤 User Behavioral Segmentation

Group users into actionable behavioral cohorts aligned with recovery outreach strategies.

In [ ]:
user_features = df.groupby('user_id').agg(
    total_events=('event_type', 'count'),
    total_views=('event_type', lambda x: (x == 'view').sum()),
    total_carts=('event_type', lambda x: (x == 'cart').sum()),
    total_purchases=('event_type', lambda x: (x == 'purchase').sum()),
    total_removes=('event_type', lambda x: (x == 'remove_from_cart').sum()),
    unique_sessions=('user_session', 'nunique'),
    unique_products_viewed=('product_id', 'nunique'),
    avg_price_viewed=('price', 'mean'),
    max_price=('price', 'max'),
    unique_categories=('category_main', 'nunique'),
    unique_brands=('brand', 'nunique'),
    first_event=('event_time', 'min'),
    last_event=('event_time', 'max'),
).reset_index()

user_features['days_active'] = (user_features['last_event'] - user_features['first_event']).dt.days + 1
user_features['view_to_cart_rate'] = (user_features['total_carts'] / user_features['total_views'].clip(lower=1) * 100).clip(0, 100)
user_features['cart_to_purchase_rate'] = np.where(
    user_features['total_carts'] > 0,
    user_features['total_purchases'] / user_features['total_carts'] * 100,
    0.0,
).clip(0, 100)
user_features['events_per_session'] = user_features['total_events'] / user_features['unique_sessions'].clip(lower=1)
display(user_features.describe().round(2))

In [ ]:
def segment_user(row):
    if row['total_purchases'] > 0 and row['total_carts'] > 0:
        return 'Loyal Buyer' if row['cart_to_purchase_rate'] > 50 else 'Selective Buyer'
    if row['total_carts'] > 0 and row['total_purchases'] == 0:
        return 'Hesitant Shopper' if row['unique_sessions'] > 3 else 'Cart Abandoner'
    if row['total_views'] > 10:
        return 'Window Shopper'
    return 'Passive Browser'

user_features['segment'] = user_features.apply(segment_user, axis=1)
segment_summary = user_features.groupby('segment').agg(
    user_count=('user_id', 'count'),
    avg_views=('total_views', 'mean'),
    avg_carts=('total_carts', 'mean'),
    avg_purchases=('total_purchases', 'mean'),
    avg_sessions=('unique_sessions', 'mean'),
).round(2)
segment_summary['share_pct'] = (segment_summary['user_count'] / len(user_features) * 100).round(1)
display(segment_summary.sort_values('user_count', ascending=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
seg_counts = user_features['segment'].value_counts()
axes[0].pie(
    seg_counts.values,
    labels=seg_counts.index,
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops=dict(width=0.5, edgecolor='white'),
)
axes[0].set_title('User Segment Proportions', fontweight='bold')

segment_summary[['avg_views', 'avg_carts', 'avg_purchases']].plot(kind='bar', ax=axes[1])
axes[1].set_title('Average Funnel Activity per Segment', fontweight='bold')
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## 11. 🔗 Session Depth & Engagement Analysis

Evaluate how session length, product breadth, and dwell duration differ across browsing, abandoned, and converted sessions.

In [ ]:
session_features = df.groupby('user_session').agg(
    user_id=('user_id', 'first'),
    event_count=('event_type', 'count'),
    unique_products=('product_id', 'nunique'),
    unique_categories=('category_main', 'nunique'),
    has_cart=('event_type', lambda x: (x == 'cart').any()),
    has_purchase=('event_type', lambda x: (x == 'purchase').any()),
    session_start=('event_time', 'min'),
    session_end=('event_time', 'max'),
).reset_index()

session_features['session_duration_min'] = (
    (session_features['session_end'] - session_features['session_start']).dt.total_seconds() / 60.0
)
session_features['outcome'] = np.where(
    session_features['has_purchase'],
    'Purchased',
    np.where(session_features['has_cart'], 'Abandoned Cart', 'Browse Only'),
)
display(session_features['outcome'].value_counts().to_frame('session_count'))

In [ ]:
outcome_metrics = session_features.groupby('outcome').agg(
    avg_events=('event_count', 'mean'),
    avg_products=('unique_products', 'mean'),
    avg_duration_min=('session_duration_min', 'mean'),
    avg_categories=('unique_categories', 'mean'),
).round(2)
display(outcome_metrics)

outcome_metrics[['avg_events', 'avg_products', 'avg_duration_min']].plot(kind='bar', figsize=(11, 5))
plt.title('Session Engagement Characteristics by Outcome', fontweight='bold')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 12. ⚙️ Feature Engineering & Target Correlation Preview

> **Note on Production Pipeline:** In production (`src/data_prep/rees46_processor.py`), features are extracted strictly at each abandoned session timestamp $T_{\text{abandon}}$ to eliminate temporal leakage. Below we inspect feature correlations against 7-day purchase recovery.

In [ ]:
cart_u = df[df['event_type'] == 'cart'].groupby('user_id').agg(
    first_cart_time=('event_time', 'min'),
    total_cart_value=('price', 'sum'),
    unique_products_carted=('product_id', 'nunique'),
).reset_index()

purch_u = df[df['event_type'] == 'purchase'].groupby('user_id').agg(
    first_purchase_time=('event_time', 'min'),
).reset_index()

label_df = cart_u.merge(purch_u, on='user_id', how='left')
label_df['days_to_purchase'] = (
    (label_df['first_purchase_time'] - label_df['first_cart_time']).dt.total_seconds() / 86400.0
)
label_df['purchased_within_7d'] = (
    (label_df['days_to_purchase'] >= 0) & (label_df['days_to_purchase'] <= 7)
).fillna(False).astype(int)

print('Target Distribution (purchased_within_7d):')
print(label_df['purchased_within_7d'].value_counts())

In [ ]:
ml_preview = user_features[user_features['total_carts'] > 0].merge(
    label_df[['user_id', 'purchased_within_7d', 'total_cart_value', 'unique_products_carted']],
    on='user_id',
    how='left',
)

feature_cols = [
    'total_views', 'total_carts', 'total_removes',
    'unique_sessions', 'unique_products_viewed', 'avg_price_viewed',
    'max_price', 'unique_categories', 'unique_brands',
    'days_active', 'view_to_cart_rate', 'events_per_session',
    'total_cart_value', 'unique_products_carted',
]
display(ml_preview[feature_cols].describe().round(2))

In [ ]:
corr_matrix = ml_preview[feature_cols + ['purchased_within_7d']].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, square=True, linewidths=0.5, ax=ax,
)
ax.set_title('Feature Correlation Matrix (with purchased_within_7d)', fontweight='bold')
plt.tight_layout()
plt.show()

## 13. 💾 Running the Production Leakage-Free Exporter

Invoke `src.data_prep.feature_engineering` directly from the notebook (or CLI) to generate schema-validated Parquet splits matching `configs/feature_schema.yaml`.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path('..').resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print('To build production schema-validated Parquet splits, run:')
print('python -m src.data_prep.feature_engineering --raw-dir data/raw --output-dir data/processed/ml')

## 14. 🎯 Key Insights & Modeling Takeaways

| Observation | Impact on Downstream Models |
|---|---|
| **High session-level abandonment (~70–80%)** | Strong class imbalance on `purchased_within_7d` — enable `is_unbalance=True` / `scale_pos_weight` in LightGBM & XGBoost. |
| **Clear diurnal & weekday conversion cycles** | Include cyclic hour/day features in both `IntentClassifier` and CoxPH `TimingOptimizer`. |
| **Nullable `brand` & `category_code`** | Aggregate count of non-empty distinct brands/categories per session rather than raw high-cardinality IDs. |
| **Repeat visitors drive recovery** | `prior_sessions_count` and `prior_conversion_rate` (computed strictly prior to $T_{\text{abandon}}$) provide strong purchase intent lift. |

---
**Next Notebook:** `02_hillstrom_criteo_eda.ipynb` — RCT Uplift Analysis & Tiki Catalog Exploration.